# Tuning Optuna do TabM no Kaggle

Este notebook roda a busca de hiperparâmetros do TabM com Optuna nos 9 datasets do TabArena, aproveitando a CPU/GPU gratuita do Kaggle.

**Como usar:**
1. Crie um novo notebook no Kaggle.
2. Copie ou faça upload deste arquivo `.ipynb`.
3. Na célula de configuração abaixo, ajuste `N_TRIALS` conforme o tempo disponível (padrão: 50).
4. Execute todas as células em ordem (`Run All`).
5. Baixe os arquivos `tuning_results_kaggle.csv` e `tuning_trials_kaggle.csv` da aba **Output** do Kaggle.
6. Coloque esses arquivos na pasta `results/` do repositório local.

In [ ]:
# ── 1. Setup: detecta ambiente, instala pacotes e clona o repositório ──────────
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/working').exists()

if IS_KAGGLE:
    # Instala pacotes que não vêm pré-instalados no Kaggle
    subprocess.run(
        ['pip', 'install', '-q', 'pytabkit[models]', 'openml', 'optuna'],
        check=True
    )

    # Clona o repositório do grupo
    REPO_URL = 'https://github.com/aanasc4/projeto-am.git'
    REPO_DIR = Path('/kaggle/working/projeto-am')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        print(f'Repositório clonado em {REPO_DIR}')
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
        print(f'Repositório atualizado em {REPO_DIR}')

    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    RESULTS_DIR = Path('/kaggle/working/results')
    CACHE_DIR   = Path('/kaggle/working/cache/tabarena')
    os.environ['TABARENA_CACHE'] = str(CACHE_DIR)
else:
    # Ambiente local: usa o layout padrão do repositório
    sys.path.insert(0, str(Path('..').resolve()))
    RESULTS_DIR = Path('../results')
    CACHE_DIR   = Path('../cache/tabarena')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ambiente : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Resultados: {RESULTS_DIR}')

In [ ]:
# ── 2. Configuração do experimento ─────────────────────────────────────────────

N_TRIALS  = 50   # número de trials Optuna por dataset (aumente se tiver mais tempo)
CV_FOLDS  = 5    # folds na validação cruzada
SEED      = 42

# Datasets por regime (task IDs do OpenML)
TASK_IDS_SMALL  = [10101, 37, 11]          # blood-transfusion, diabetes, balance-scale
TASK_IDS_MEDIUM = [31, 9952, 23]           # credit-g, phoneme, cmc
TASK_IDS_LARGE  = [7592, 14965, 146195]    # adult, bank-marketing, connect-4
TASK_IDS_ALL    = TASK_IDS_SMALL + TASK_IDS_MEDIUM + TASK_IDS_LARGE

print(f'N_TRIALS={N_TRIALS} | CV_FOLDS={CV_FOLDS} | SEED={SEED}')
print(f'Datasets: {len(TASK_IDS_ALL)} (small={len(TASK_IDS_SMALL)}, '
      f'medium={len(TASK_IDS_MEDIUM)}, large={len(TASK_IDS_LARGE)})')

In [ ]:
# ── 3. Imports ─────────────────────────────────────────────────────────────────
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suprime logs verbosos

from data.load_tabarena import load_task
from src.pipeline.split import stratified_split
from src.pipeline.tune import tabm_factory, tabm_search_space, tune

In [ ]:
# ── 4. Tuning Optuna nos 9 datasets ────────────────────────────────────────────
best_rows  = []
trial_rows = []

for task_id in TASK_IDS_ALL:
    ds = load_task(task_id)
    X_train, _, y_train, _ = stratified_split(ds.X, ds.y, seed=SEED)

    print(f'\n[{ds.name}]  n={ds.n_samples}, regime={ds.regime}, classes={ds.n_classes}')
    print(f'  Tuning: {N_TRIALS} trials × {CV_FOLDS}-fold CV ...')

    best_params, best_auc, study = tune(
        estimator_factory=tabm_factory,
        search_space=tabm_search_space,
        X=X_train,
        y=y_train,
        seed=SEED,
        n_trials=N_TRIALS,
        cv_folds=CV_FOLDS,
    )
    print(f'  Melhor AUC (CV)={best_auc:.4f} | params={best_params}')

    best_rows.append({
        'task_id'    : task_id,
        'dataset'    : ds.name,
        'regime'     : ds.regime,
        'best_auc_cv': best_auc,
        **best_params,
    })

    for t in study.trials:
        trial_rows.append({
            'task_id': task_id,
            'dataset': ds.name,
            'trial'  : t.number,
            'auc_cv' : t.value,
            **t.params,
        })

print('\nTuning concluído!')

In [ ]:
# ── 5. Salvar resultados ────────────────────────────────────────────────────────
df_best   = pd.DataFrame(best_rows)
df_trials = pd.DataFrame(trial_rows)

best_path   = RESULTS_DIR / 'tuning_results_kaggle.csv'
trials_path = RESULTS_DIR / 'tuning_trials_kaggle.csv'

df_best.to_csv(best_path,   index=False)
df_trials.to_csv(trials_path, index=False)

print(f'Salvo: {best_path}')
print(f'Salvo: {trials_path}')
df_best

In [ ]:
# ── 6. Resumo por regime ────────────────────────────────────────────────────────
print('Resumo AUC (CV) por regime:')
print(
    df_best.groupby('regime')['best_auc_cv']
           .agg(['mean', 'min', 'max', 'count'])
           .round(4)
           .to_string()
)

print('\nMelhores hiperparâmetros por dataset:')
print(df_best[['dataset', 'regime', 'best_auc_cv', 'num_emb_type', 'tabm_k', 'n_epochs']].to_string(index=False))